# DIC予測 v1：2つの戦略を同じ条件で比べてから提出する

方針は戦略メモの「1周目」タブに書いたとおり。このノートでは次の順に進める。

1. データ確認：列名をそろえ（`TA1.x`→`TA1`）、欠損・重複列・id順を確認する
2. 候補を同じ分割で比較：ChatGPT案のK・M、Claude案のCL、それらの固定比率の平均
3. 補助評価：同じ緯度・経度を同じfoldにまとめた分割（未知地点への弱さの確認）
4. 事前に決めたルールで採用案を選び、全train（1,454行）で再学習して `submission.csv` を作る

守ること
- モデルは全結合NNだけ。前処理は列の選択、平均・標準偏差、四則演算、log1p、clipだけ（ドメイン特徴・交差特徴は使わない）
- 平均・標準偏差・外れ値判定・クリップ範囲は、すべて学習部分だけから求める
- 評価は元のDIC単位のRMSE。各行を学習に使わなかったモデルの予測（OOF）を全1,454行分集めて計算する
- 評価用のfoldは学習にも学習回数の決定にも使わない（全候補とも学習回数は固定）

実行時間の目安：CPU 2コアで主比較が約8分、補助評価が約1分、全件学習が約20秒。`QUICK = True` にすると分割1通り・seed1つで試せる。

保存されている出力は、Claudeの作業環境（Linux、CPU 2コア、PyTorch 2.15.0）でこのノートのコードをそのまま実行したもの（2026年11月17日）。Colabで再実行すると、ライブラリの版によって小数点以下が少し変わることがある。

方針：[戦略メモ「1周目」タブ](https://docs.google.com/document/d/1U_WHC9poxdGDOFZp6pdwKSpunfI8yoDc2RCIthOKS2M/edit?tab=t.3hy9e1rjtymd)

In [ ]:
# 0. 設定・データ読み込み（GitHubのRawデータから読み込みます）
import os, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

QUICK = False          # True: 分割seed1通り×学習seed1つで動作確認だけ行う
RUN_GROUP_CV = True    # 補助評価（座標ごとの分割）を実行するか

OUT_DIR = '/content/dic_v1'
os.makedirs(OUT_DIR, exist_ok=True)

# GitHubのRaw URL
BASE_URL = 'https://raw.githubusercontent.com/Toyoda05/Kaggle-eds232/main/'

train = pd.read_csv(BASE_URL + 'train.csv')
test = pd.read_csv(BASE_URL + 'test.csv')
sub = pd.read_csv(BASE_URL + 'sample_submission.csv')

torch.set_num_threads(max(1, os.cpu_count() or 1))
print('torch', torch.__version__, '| CPU', os.cpu_count())
print('train', train.shape, 'test', test.shape, 'sample', sub.shape)

torch 2.14.1+cu130 | CPU 2
train (1454, 19) test (485, 17) sample (485, 2)


In [ ]:
# 1. データ確認：列をそろえ、前提が崩れていないことを assert で確かめる
tr = train.drop(columns=['Unnamed: 12']).rename(columns={'TA1.x': 'TA1'})
te = test.copy()

INPUT_ALL = [c for c in tr.columns if c not in ('id', 'DIC')]
assert INPUT_ALL == [c for c in te.columns if c != 'id'], 'trainとtestの入力列（名前・順序）が一致しません'
assert len(INPUT_ALL) == 16
assert tr[INPUT_ALL + ['DIC']].notna().all().all() and te[INPUT_ALL].notna().all().all(), '欠損があります'
assert (tr['NH3uM'] == tr['R_Nuts']).all() and (te['NH3uM'] == te['R_Nuts']).all()
assert np.array_equal(te['id'].values, sub['id'].values), 'testとsample_submissionのid順が違います'

y_all = tr['DIC'].to_numpy(dtype=np.float64)
N = len(tr)
# 戦略2（Claude）で確認した極端な2行（NO2uMが6.19・8.19）。除外はせず、参考の集計だけに使う
EXTREME_IDS = [702, 854]
is_extreme = tr['id'].isin(EXTREME_IDS).to_numpy()
# 補助評価用：緯度・経度が完全に一致する行を同じグループにする
coord_group = tr.groupby(['Lat_Dec', 'Lon_Dec'], sort=False).ngroup().to_numpy()

print('入力候補16列:', INPUT_ALL)
print('欠損なし／NH3uM=R_Nuts（全行）／id順一致 を確認')
print('DIC 平均 %.2f, 標準偏差 %.2f' % (y_all.mean(), y_all.std()))
print('極端な2行:', tr.loc[is_extreme, ['id', 'NO2uM', 'DIC']].to_dict('records'))
print('座標の数（train）:', coord_group.max() + 1)

入力候補16列: ['Lat_Dec', 'Lon_Dec', 'NO2uM', 'NO3uM', 'NH3uM', 'R_TEMP', 'R_Depth', 'R_Sal', 'R_DYNHT', 'R_Nuts', 'R_Oxy_micromol.Kg', 'PO4uM', 'SiO3uM', 'TA1', 'Salinity1', 'Temperature_degC']
欠損なし／NH3uM=R_Nuts（全行）／id順一致 を確認
DIC 平均 2150.47, 標準偏差 113.12
極端な2行: [{'id': 702, 'NO2uM': 6.19, 'DIC': 2367.8}, {'id': 854, 'NO2uM': 8.19, 'DIC': 2361.81}]
座標の数（train）: 259


## 2. 比較する候補

| 名前 | 出どころ | 入力 | 学習 |
| --- | --- | --- | --- |
| K | 戦略1（ChatGPT）の採用候補 | 15列（TA1あり・R_Nutsなし） | SiLU 128→64、MSE、Adam 1e-3固定、209 epoch |
| M | 戦略1（ChatGPT）の平均相手 | 15列（TA1なし・R_Nutsあり） | SiLU 128→64、MSE、Adam 1e-3固定、179 epoch |
| CL | 戦略2（Claude）の最終構成 | 14列（R_Nuts・R_TEMPなし）、5列をlog1p | SiLU 128→64＋線形スキップ、Huber δ=0.05、Adam 3e-3→0のcosine、300 epoch、学習側の外れ値除外、予測時のクリップ |

共通：入力とDICは学習部分の平均・標準偏差で標準化、batch 64、重み減衰・Dropoutなし。

K・Mの学習回数（209・179）は、戦略1の「早期終了で決めた最良epochの中央値」で、提出用の全件学習と同じ回数。
ここでは評価用foldで早期終了をしないため、戦略1の記録（評価foldで早期終了）より少し悪い値が出る可能性がある。

予測の平均（重みは固定・探索しない）：`K+M`（戦略1の最終版）、`CL`、`CL+K+M`（CLを1/2、K・Mを1/4ずつ）。

In [ ]:
# 2. 候補の定義
COMMON = dict(hidden=(128, 64), batch=64)
CONFIGS = {
    'K':  dict(drop=['R_Nuts'], log1p=[], loss='mse', lr=1e-3, schedule='const', epochs=209,
               outlier_k=None, clip=False, skip=False),
    'M':  dict(drop=['TA1'], log1p=[], loss='mse', lr=1e-3, schedule='const', epochs=179,
               outlier_k=None, clip=False, skip=False),
    'CL': dict(drop=['R_Nuts', 'R_TEMP'], log1p=['NO2uM', 'NH3uM', 'R_Depth', 'R_DYNHT', 'SiO3uM'],
               loss='huber', delta=0.05, lr=3e-3, schedule='cosine', epochs=300,
               outlier_k=6.0, clip=True, skip=True),
}
BLENDS = {
    'K+M':    {'K': 0.5, 'M': 0.5},
    'CL':     {'CL': 1.0},
    'CL+K+M': {'CL': 0.5, 'K': 0.25, 'M': 0.25},
}
SPLIT_SEEDS = [2026, 2027, 100]   # 2026・2027は戦略1、100は戦略2で使った分割seed
TRAIN_SEEDS = [17, 42, 2026]      # 学習seed（全候補で同じ3つ）
if QUICK:
    SPLIT_SEEDS, TRAIN_SEEDS = [2026], [42]

def feature_cols(cfg):
    return [c for c in INPUT_ALL if c not in cfg['drop']]

def make_X(df, cfg):
    X = df[feature_cols(cfg)].astype(np.float64).copy()
    for c in cfg['log1p']:
        X[c] = np.log1p(X[c])
    return X.to_numpy()

for name, cfg in CONFIGS.items():
    print(f'{name}: {len(feature_cols(cfg))}列', feature_cols(cfg))

K: 15列 ['Lat_Dec', 'Lon_Dec', 'NO2uM', 'NO3uM', 'NH3uM', 'R_TEMP', 'R_Depth', 'R_Sal', 'R_DYNHT', 'R_Oxy_micromol.Kg', 'PO4uM', 'SiO3uM', 'TA1', 'Salinity1', 'Temperature_degC']
M: 15列 ['Lat_Dec', 'Lon_Dec', 'NO2uM', 'NO3uM', 'NH3uM', 'R_TEMP', 'R_Depth', 'R_Sal', 'R_DYNHT', 'R_Nuts', 'R_Oxy_micromol.Kg', 'PO4uM', 'SiO3uM', 'Salinity1', 'Temperature_degC']
CL: 14列 ['Lat_Dec', 'Lon_Dec', 'NO2uM', 'NO3uM', 'NH3uM', 'R_Depth', 'R_Sal', 'R_DYNHT', 'R_Oxy_micromol.Kg', 'PO4uM', 'SiO3uM', 'TA1', 'Salinity1', 'Temperature_degC']


In [ ]:
# 3. モデル・学習関数
class MLP(nn.Module):
    """全結合NN：Linear→SiLU→Linear→SiLU→Linear（出力は活性化なし）。skip=Trueで入力→出力の線形層を並列に足す"""
    def __init__(self, n_in, hidden=(128, 64), skip=False):
        super().__init__()
        layers, d = [], n_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.SiLU()]
            d = h
        layers.append(nn.Linear(d, 1))
        self.body = nn.Sequential(*layers)
        self.skip = None
        if skip:
            self.skip = nn.Linear(n_in, 1)
            nn.init.zeros_(self.skip.weight)
            nn.init.zeros_(self.skip.bias)

    def forward(self, x):
        out = self.body(x)
        if self.skip is not None:
            out = out + self.skip(x)
        return out.squeeze(-1)


def outlier_mask(X, y, k):
    """学習部分だけで線形回帰（最小二乗）を当て、残差が頑健な標準偏差のk倍以内の行をTrueにする"""
    Xs = (X - X.mean(0)) / (X.std(0) + 1e-9)
    A = np.c_[Xs, np.ones(len(Xs))]
    w = np.linalg.lstsq(A, y, rcond=None)[0]
    r = y - A @ w
    robust_sd = 1.4826 * np.median(np.abs(r - np.median(r)))
    return np.abs(r) <= k * robust_sd


def fit_predict(cfg, Xtr, ytr, X_list, seed):
    """学習部分(Xtr, ytr)だけで前処理値を求めて学習し、X_listの各行列を元のDIC単位で予測する"""
    if cfg['outlier_k'] is not None:
        keep = outlier_mask(Xtr, ytr, cfg['outlier_k'])
        Xtr, ytr = Xtr[keep], ytr[keep]
    mu, sd = Xtr.mean(0), Xtr.std(0)
    sd = np.where(sd == 0, 1.0, sd)
    ym, ys = ytr.mean(), ytr.std()
    Xt = torch.tensor((Xtr - mu) / sd, dtype=torch.float32)
    yt = torch.tensor((ytr - ym) / ys, dtype=torch.float32)

    torch.manual_seed(seed)
    model = MLP(Xt.shape[1], COMMON['hidden'], cfg['skip'])
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    batch, epochs = COMMON['batch'], cfg['epochs']
    steps = int(np.ceil(len(Xt) / batch))
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * steps)
             if cfg['schedule'] == 'cosine' else None)
    crit = nn.HuberLoss(delta=cfg['delta']) if cfg['loss'] == 'huber' else nn.MSELoss()
    g = torch.Generator().manual_seed(seed)
    model.train()
    for _ in range(epochs):
        idx = torch.randperm(len(Xt), generator=g)
        for s in range(steps):
            b = idx[s * batch:(s + 1) * batch]
            opt.zero_grad()
            loss = crit(model(Xt[b]), yt[b])
            loss.backward()
            opt.step()
            if sched is not None:
                sched.step()
    model.eval()
    lo, hi = Xt.min(0).values.numpy(), Xt.max(0).values.numpy()
    preds = []
    with torch.no_grad():
        for X in X_list:
            Z = (X - mu) / sd
            if cfg['clip']:
                Z = np.clip(Z, lo, hi)  # 学習部分の範囲外へ外挿しない
            preds.append(model(torch.tensor(Z, dtype=torch.float32)).numpy().astype(np.float64) * ys + ym)
    return preds


def rmse(p, y=y_all, mask=None):
    if mask is not None:
        p, y = p[mask], y[mask]
    return float(np.sqrt(np.mean((p - y) ** 2)))

# 動作確認（1回だけ学習して時間を測る）
_t = time.time()
_cfg = CONFIGS['CL']
_X = make_X(tr, _cfg)
_p = fit_predict(_cfg, _X[:1163], y_all[:1163], [_X[1163:]], seed=0)[0]
print('動作確認 RMSE %.3f / %.1f秒' % (rmse(_p, y_all[1163:]), time.time() - _t))

動作確認 RMSE 5.087 / 19.3秒


In [ ]:
# 4. 分割と交差検証（OOF予測はファイルに保存し、途中で止まっても再開できる）
def make_folds(split_seed, groups=None, k=5):
    """各行のfold番号を返す。groupsを渡すと同じグループは必ず同じfoldに入る"""
    rng = np.random.RandomState(split_seed)
    fold = np.empty(N, dtype=int)
    if groups is None:
        for f, part in enumerate(np.array_split(rng.permutation(N), k)):
            fold[part] = f
    else:
        names = np.unique(groups)
        for f, part in enumerate(np.array_split(rng.permutation(names), k)):
            fold[np.isin(groups, part)] = f
    return fold


def run_cv(name, split_seed, seeds, split_type='random'):
    """候補nameのOOF予測（学習seedごと）を返す。shape = (len(seeds), N)"""
    cfg = CONFIGS[name]
    X = make_X(tr, cfg)
    fold = make_folds(split_seed, coord_group if split_type == 'group' else None)
    out = np.zeros((len(seeds), N))
    for i, seed in enumerate(seeds):
        path = os.path.join(OUT_DIR, f'oof_{split_type}{split_seed}_{name}_seed{seed}.npy')
        if os.path.exists(path):
            out[i] = np.load(path)
            continue
        for f in range(fold.max() + 1):
            a, b = fold != f, fold == f
            out[i, b] = fit_predict(cfg, X[a], y_all[a], [X[b]], seed=seed * 100 + f)[0]
        np.save(path, out[i])
    return out


def blend(oofs, weights):
    """各候補の学習seed平均を、固定の重みで混ぜる"""
    return sum(w * oofs[n].mean(0) for n, w in weights.items())

In [ ]:
# 5. 主評価：3通りのランダム5分割 × 学習seed3つで、K・M・CLのOOFを作る
t0 = time.time()
OOF = {}
for ss in SPLIT_SEEDS:
    OOF[ss] = {}
    for name in CONFIGS:
        t1 = time.time()
        OOF[ss][name] = run_cv(name, ss, TRAIN_SEEDS)
        print(f'split {ss} {name:>2}: seed平均OOF RMSE {rmse(OOF[ss][name].mean(0)):.4f}'
              f'（{time.time() - t1:.0f}秒）')
print(f'合計 {time.time() - t0:.0f}秒')

split 2026  K: seed平均OOF RMSE 5.5874（110秒）
split 2026  M: seed平均OOF RMSE 5.5745（91秒）
split 2026 CL: seed平均OOF RMSE 4.9990（168秒）
split 2027  K: seed平均OOF RMSE 8.7992（114秒）
split 2027  M: seed平均OOF RMSE 11.4254（91秒）
split 2027 CL: seed平均OOF RMSE 4.9688（166秒）
split 100  K: seed平均OOF RMSE 5.7127（107秒）
split 100  M: seed平均OOF RMSE 5.5689（91秒）
split 100 CL: seed平均OOF RMSE 4.9758（168秒）
合計 1105秒


In [ ]:
# 6. 集計：候補ごと・分割ごとのOOF RMSE（全行／極端な2行を除く）
rows = []
for ss in SPLIT_SEEDS:
    for name in CONFIGS:
        for i, seed in enumerate(TRAIN_SEEDS):
            rows.append(dict(split=ss, candidate=name, kind=f'単体 seed{seed}', rmse=rmse(OOF[ss][name][i]),
                             rmse_wo2=rmse(OOF[ss][name][i], mask=~is_extreme)))
    for bname, w in BLENDS.items():
        p = blend(OOF[ss], w)
        rows.append(dict(split=ss, candidate=bname, kind='平均（3seed）', rmse=rmse(p), rmse_wo2=rmse(p, mask=~is_extreme)))
res = pd.DataFrame(rows)

table = (res[res.kind == '平均（3seed）']
         .pivot(index='candidate', columns='split', values='rmse')[SPLIT_SEEDS])
table['平均'] = table[SPLIT_SEEDS].mean(axis=1)
table_wo2 = (res[res.kind == '平均（3seed）']
             .pivot(index='candidate', columns='split', values='rmse_wo2')[SPLIT_SEEDS])
table_wo2['平均'] = table_wo2[SPLIT_SEEDS].mean(axis=1)
single = (res[res.kind != '平均（3seed）'].groupby(['candidate', 'split']).rmse.mean()
          .unstack()[SPLIT_SEEDS])
single['平均'] = single[SPLIT_SEEDS].mean(axis=1)

print('【主指標】OOF RMSE（全1,454行、3seedの予測平均、元のDIC単位）')
display(table.loc[list(BLENDS)].round(4))
print('【参考】極端な2行（id 702・854）を除いたRMSE')
display(table_wo2.loc[list(BLENDS)].round(4))
print('【参考】単体モデル（1seed）のOOF RMSEの平均')
display(single.round(4))
res.to_csv(os.path.join(OUT_DIR, 'cv_results.csv'), index=False)

【主指標】OOF RMSE（全1,454行、3seedの予測平均、元のDIC単位）


split,2026,2027,100,平均
candidate,,,,
K+M,5.5343,10.0545,5.5696,7.0528
CL,4.9990,4.9688,4.9758,4.9812
CL+K+M,5.1072,6.6277,5.1404,5.6251


【参考】極端な2行（id 702・854）を除いたRMSE


split,2026,2027,100,平均
candidate,,,,
K+M,5.2152,5.2229,5.2953,5.2445
CL,4.9917,4.9711,4.9702,4.9777
CL+K+M,5.0092,5.0224,5.0529,5.0281


【参考】単体モデル（1seed）のOOF RMSEの平均


split,2026,2027,100,平均
candidate,,,,
CL,5.0066,4.9755,4.9842,4.9888
K,5.8135,9.1480,5.9265,6.9627
M,5.6918,11.6570,5.7489,7.6992


In [ ]:
# 7. 採用ルール（実行前に決めたもの）
#   (1) 主指標（全行OOF RMSE）の分割平均が最も小さい候補を暫定1位にする
#   (2) 1位とモデル数がより少ない候補との差が0.02未満なら、少ない方を採用する（モデル数 CL:3 < K+M:6 < CL+K+M:9）
#   (3) 2行除外RMSEは採否に使わず、全行の差が極端な2行だけで生じていないかの確認に使う
N_MODELS = {'CL': 3, 'K+M': 6, 'CL+K+M': 9}
MARGIN = 0.02
order = table.loc[list(BLENDS), '平均'].sort_values()
best = order.index[0]
adopted = best
for cand in sorted(N_MODELS, key=N_MODELS.get):
    if N_MODELS[cand] < N_MODELS[adopted] and order[cand] - order[best] < MARGIN:
        adopted = cand
        break
print('分割平均の順位:', order.round(4).to_dict())
print('採用:', adopted, '| 各分割で採用案が最良だった回数:',
      int((table.loc[list(BLENDS), SPLIT_SEEDS].idxmin() == adopted).sum()), '/', len(SPLIT_SEEDS))

分割平均の順位: {'CL': 4.9812, 'CL+K+M': 5.6251, 'K+M': 7.0528}
採用: CL | 各分割で採用案が最良だった回数: 3 / 3


In [ ]:
# 8. 補助評価：同じ緯度・経度の行を同じfoldにまとめた5分割（分割seed 2026・学習seed 42）
#    未知地点への弱さを見るための値。主指標とは平均せず、採否にも使わない
if RUN_GROUP_CV:
    t0 = time.time()
    G = {name: run_cv(name, 2026, [42], split_type='group') for name in CONFIGS}
    rand = {name: OOF[SPLIT_SEEDS[0]][name][TRAIN_SEEDS.index(42) if 42 in TRAIN_SEEDS else 0] for name in CONFIGS}
    rows_g = []
    for bname, w in BLENDS.items():
        pg = sum(wt * G[n][0] for n, wt in w.items())
        pr = sum(wt * rand[n] for n, wt in w.items())
        rows_g.append(dict(candidate=bname, random_rmse=rmse(pr), group_rmse=rmse(pg),
                           group_rmse_wo2=rmse(pg, mask=~is_extreme)))
    group_table = pd.DataFrame(rows_g).set_index('candidate')
    print(f'学習seed 42のみ。ランダム分割は分割seed {SPLIT_SEEDS[0]}（{time.time() - t0:.0f}秒）')
    display(group_table.round(4))

学習seed 42のみ。ランダム分割は分割seed 2026（128秒）


,random_rmse,group_rmse,group_rmse_wo2
candidate,,,
K+M,5.6478,5.7137,5.4331
CL,5.0048,5.0422,5.0368
CL+K+M,5.1309,5.2105,5.1261


## 3. 提出ファイルの作成

採用案に含まれる各候補を、全train 1,454行・学習seed 3つで学習し直す（学習回数は上の固定値のまま。検証用を分けないので早期終了は使わない）。
前処理値（平均・標準偏差・外れ値判定・クリップ範囲）も全trainから求め直す。元のDIC単位に戻した予測を、採用案の重みで平均する。

In [ ]:
# 9. 全trainで再学習して submission.csv を作る
t0 = time.time()
test_pred = {}
for name in BLENDS[adopted]:
    cfg = CONFIGS[name]
    X, Xte = make_X(tr, cfg), make_X(te, cfg)
    test_pred[name] = np.mean([fit_predict(cfg, X, y_all, [Xte], seed=seed)[0] for seed in [17, 42, 2026]], axis=0)
pred = sum(w * test_pred[n] for n, w in BLENDS[adopted].items())

submission = pd.DataFrame({'id': te['id'].values, 'DIC': pred})
assert len(submission) == 485 and submission.columns.tolist() == ['id', 'DIC']
assert submission['DIC'].notna().all() and np.isfinite(submission['DIC']).all()
assert np.array_equal(submission['id'].values, sub['id'].values)
submission.to_csv('submission.csv', index=False)

record = dict(adopted=adopted, weights=BLENDS[adopted], configs={n: CONFIGS[n] for n in BLENDS[adopted]},
              split_seeds=SPLIT_SEEDS, train_seeds=TRAIN_SEEDS,
              oof_rmse={str(k): round(v, 5) for k, v in table.loc[adopted, SPLIT_SEEDS].items()},
              oof_rmse_mean=round(float(table.loc[adopted, '平均']), 5))
with open(os.path.join(OUT_DIR, 'selection.json'), 'w', encoding='utf-8') as fp:
    json.dump(record, fp, ensure_ascii=False, indent=2)
print(json.dumps(record, ensure_ascii=False, indent=1))
print(f'全件学習 {time.time() - t0:.0f}秒 | test予測 平均 {pred.mean():.2f}, 範囲 {pred.min():.2f}〜{pred.max():.2f}')
display(submission.head())
# ダウンロードする場合
from google.colab import files; files.download('submission.csv')

{
 "adopted": "CL",
 "weights": {
  "CL": 1.0
 },
 "configs": {
  "CL": {
   "drop": [
    "R_Nuts",
    "R_TEMP"
   ],
   "log1p": [
    "NO2uM",
    "NH3uM",
    "R_Depth",
    "R_DYNHT",
    "SiO3uM"
   ],
   "loss": "huber",
   "delta": 0.05,
   "lr": 0.003,
   "schedule": "cosine",
   "epochs": 300,
   "outlier_k": 6.0,
   "clip": true,
   "skip": true
  }
 },
 "split_seeds": [
  2026,
  2027,
  100
 ],
 "train_seeds": [
  17,
  42,
  2026
 ],
 "oof_rmse": {
  "2026": 4.99896,
  "2027": 4.96883,
  "100": 4.97582
 },
 "oof_rmse_mean": 4.9812
}
全件学習 44秒 | test予測 平均 2156.87, 範囲 1979.40〜2356.10


,id,DIC
0,1455,2170.421896
1,1456,2194.759407
2,1457,2327.992820
3,1458,1989.670671
4,1459,2149.106665


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# score: （Kaggle未提出）内部OOF RMSE 4.981（CL・3seed平均、ランダム5分割×分割seed 2026/2027/100の平均）
# name: 杉浦孔明
# id: 123456